# Excercises 

## 0. Setup your own repo

- You can add `mads_datasets` and `mltrainer` as dependencies to your own repo, in addition to more basic things like jupyter, torch and seaborn.
- If you want to use the tomlserializer, add `tomlserializer` as a dependency. For tensorboard, add `tensorboard` and `torch-tb-profiler`.
- Invite me (raoulg; https://github.com/raoulg) as a collaborator to your repo.



In [23]:
from mads_datasets import DatasetFactoryProvider, DatasetType

from mltrainer.preprocessors import BasePreprocessor
from mltrainer import imagemodels, Trainer, TrainerSettings, ReportTypes, metrics

import torch.optim as optim
from torch import nn
from tomlserializer import TOMLSerializer

In [24]:
import mlflow


In [25]:
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
preprocessor = BasePreprocessor()
streamers = fashionfactory.create_datastreamer(batchsize=64, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()

2026-03-30 15:46:04.621 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\D110303\.cache\mads_datasets\fashionmnist
2026-03-30 15:46:04.622 | INFO     | mads_datasets.base:download_data:124 - File already exists at C:\Users\D110303\.cache\mads_datasets\fashionmnist\fashionmnist.pt


In [26]:
accuracy = metrics.Accuracy()

In [27]:
import torch

loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=100,
    valid_steps=100,
    reporttypes=[ReportTypes.MLFLOW, ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


In [28]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2

        # changing to convulutional, adding conv layer and pooling exc4
        self.conv_stack = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),   # 28x28 → 14x14

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),   # 14x14 → 7x7
        )

        self.flatten = nn.Flatten()

        # secondpart
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(32 * 7 * 7, units1),
            nn.BatchNorm1d(units1),
            nn.ReLU(),
            nn.Dropout(p=0.1),

            nn.Linear(units1, units2),
            nn.BatchNorm1d(units2),
            nn.ReLU(),
            nn.Dropout(p=0.1),

            nn.Linear(units2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_stack(x)
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork(
    num_classes=10, units1=256, units2=256
)

In [29]:
{k: v for k, v in model.__dict__.items() if not k.startswith("_")}

{'training': True, 'num_classes': 10, 'units1': 256, 'units2': 256}

This means that if you want to add more parameters to the `.toml` file, eg `units3`, you can add them to the class like this:

```python
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int, units3: int) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.units3 = units3  # <-- add this line
```

And then it will be added to the `.toml` file. Check the result for yourself by using the `.save()` method of the `TomlSerializer` class like this:

In [30]:
tomlserializer = TOMLSerializer()
tomlserializer.save(settings, "settings.toml")
tomlserializer.save(model, "model.toml")

In [31]:
trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainstreamer,
    validdataloader=validstreamer,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau
)

trainer.loop()

2026-03-30 15:46:04.725 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs\20260330-154604
2026-03-30 15:46:04.727 | INFO     | mltrainer.trainer:__init__:66 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 100/100 [00:01<00:00, 51.31it/s]
2026-03-30 15:46:07.811 | INFO     | mltrainer.trainer:report:215 - Epoch 0 train 0.6077 test 0.4257 metric ['0.8430']
100%|██████████| 100/100 [00:03<00:00, 30.58it/s]
2026-03-30 15:46:12.199 | INFO     | mltrainer.trainer:report:215 - Epoch 1 train 0.3976 test 0.4013 metric ['0.8528']
100%|██████████| 100/100 [00:03<00:00, 32.20it/s]
2026-03-30 15:46:17.759 | INFO     | mltrainer.trainer:report:215 - Epoch 2 train 0.3581 test 0.3445 metric ['0.8711']
100%|██████████| 3/3 [00:13<00:00,  4.34s/it]


In [32]:
units = [256, 128, 64]
for unit1 in units:
    for unit2 in units:
        print(f"Units: {unit1}, {unit2}")

Units: 256, 256
Units: 256, 128
Units: 256, 64
Units: 128, 256
Units: 128, 128
Units: 128, 64
Units: 64, 256
Units: 64, 128
Units: 64, 64


In [33]:
import torch

mlflow.end_run()

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("cnn_fashion_experiment")

units = [256, 512]
loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=4,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.MLFLOW, ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

for unit1 in units:
    for unit2 in units:

        with mlflow.start_run():
            model = NeuralNetwork(
                num_classes=10,
                units1=unit1,
                units2=unit2
            )

            mlflow.log_param("units1", unit1)
            mlflow.log_param("units2", unit2)

            trainer = Trainer(
                model=model,
                settings=settings,
                loss_fn=loss_fn,
                optimizer=optim.Adam,
                traindataloader=trainstreamer,
                validdataloader=validstreamer,
                scheduler=optim.lr_scheduler.ReduceLROnPlateau
            )

            trainer.loop()


Traceback (most recent call last):
  File "c:\Users\D110303\OneDrive - EP Commodities B.V\Bureaublad\Deeplearning_ScriptsMax\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 368, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\D110303\OneDrive - EP Commodities B.V\Bureaublad\Deeplearning_ScriptsMax\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 466, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\D110303\OneDrive - EP Commodities B.V\Bureaublad\Deeplearning_ScriptsMax\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1636, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\D110303\On

2026-03-30 15:46:18.007 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs\20260330-154618
2026-03-30 15:46:18.009 | INFO     | mltrainer.trainer:__init__:66 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 937/937 [00:28<00:00, 33.02it/s]
2026-03-30 15:46:47.878 | INFO     | mltrainer.trainer:report:215 - Epoch 0 train 0.3422 test 0.2756 metric ['0.8987']
100%|██████████| 937/937 [00:41<00:00, 22.76it/s]
2026-03-30 15:47:30.427 | INFO     | mltrainer.trainer:report:215 - Epoch 1 train 0.2379 test 0.2770 metric ['0.8960']
2026-03-30 15:47:30.429 | INFO     | mltrainer.trainer:__call__:258 - best loss: 0.2756, current loss 0.2770.Counter 1/10.
100%|██████████| 937/937 [00:25<00:00, 37.23it/s]
2026-03-30 15:47:56.918 | INFO     | mltrainer.trainer:report:215 - Epoch 2 train 0.1974 test 0.2334 metric ['0.9182']
100%|██████████| 937/937 [00:24<00:00, 38.58it/s]
2026-03-30 15:48:22.524 | INFO     | mltrainer.trainer:re

In [34]:
# path for actual files tensorboard --logdir "C:\Users\D110303\OneDrive - EP Commodities B.V\Bureaublad\Deeplearning_ScriptsMax\College 2 - CNN\modellogs"

In [36]:
# voor mlflow eerst dit doen: mlflow ui --port 5000

In [ ]:
# path for actual files met mlflow mlflow ui --backend-store-uri C:\Users\D110303\OneDrive - EP Commodities B.V\Bureaublad\Deeplearning_ScriptsMax\College 2 - CNN\mlruns\273260633354334900

In [ ]:
# en dan als laatst http://127.0.0.1:5000

In [37]:
import os
print(os.getcwd())

c:\Users\D110303\OneDrive - EP Commodities B.V\Bureaublad\Deeplearning_ScriptsMax\College 2 - CNN


Run the experiment, and study the result with tensorboard. 

Locally, it is easy to do that with VS code itself. On the server, you have to take these steps:

- in the terminal, `cd` to the location of the repository
- activate the python environment for the shell. Note how the correct environment is being activated.
- run `tensorboard --logdir=modellogs` in the terminal
- tensorboard will launch at `localhost:6006` and vscode will notify you that the port is forwarded
- you can either press the `launch` button in VScode or open your local browser at `localhost:6006`


# Report
## 1. experiment
Experiment with things like:
- change the number of epochs, eg to 5 or 10. 
- changing the amount of units1 and units2 to values between 16 and 1024. Use factors of 2 to easily scan the ranges: 16, 32, 64, etc.
- changing the batchsize to values between 4 and 128. Again, use factors of two for convenience.
- change the depth of your model by adding a additional linear layer + activation function
- changing the learningrate to values between 1e-2 and 1e-5 
- changing the optimizer from SGD to one of the other available algoritms at [torch](https://pytorch.org/docs/stable/optim.html) (scroll down for the algorithms)

Check the results:
- all your experiments are saved in the `modellogs` directory, with a timestamp. Inside you find a model.toml file, that 
contains all the settings of the model. The `events` file is what tensorboard will show.
- visualize the relationship between variables: for example, make a heatmap of units vs layers.

Studyquestions:
- Epochs: what is the upside, what is the downside of increasing epochs? Do you need more epochs to find out which configuration is best? When do you need that, when not?
- what is an upside of using factors of 2 for hypertuning? What is a downside?

## Note
A note on train_steps: this is a setting that determines how often you get an update. 
Because our complete dataset is 938 (60000 / 64) batches long, you will need 938 trainstep to cover the complete 60.000 images.

This can actually be a bit confusion for some students, because changing the value of trainsteps 938 changes the meaning of an `epoch` slightly, because one epoch is no longer the full dataset, but simply `trainstep` batches. Setting trainsteps to 100 means you need to wait twice as long before you get feedback on the performance, as compared to trainsteps=50. You will see that settings trainsteps to 100 improves the learning, but that is simply because the model has seen twice as much examples as compared to trainsteps=50.

This implies that it is not usefull to compare trainsteps=50 and trainsteps=100, because setting it to 100 will always be better.
Just pick an amount that works for your hardware & patience, and adjust your number of epochs accordingly (increase epochs with lower values for trainsteps)

# 2. Reflect
Doing a master means you don't just start engineering a pipeline, but you need to reflect. Why do you see the results you see? What does this mean, considering the theory? Write down lessons learned and reflections, based on experimental results. This is the `science` part of `data science`.

You follow this cycle:
- make a hypothesis
- design an experiment
- run the experiment
- analyze the results and draw conclusions
- repeat


# 3. Make a short report
Make a short 1 a4 page report of your findings.
pay attention to:
- what was your hypothesis about interaction between hyperparameters?
- what did you find?
- visualise your results about the relationship between hyperparameters.
